In [10]:
from datasets import load_dataset
from datasets.exceptions import DatasetNotFoundError

task_names = [
    "task075_squad1.1_answer_generation",
    "task568_circa_question_generation",
    
	"task1291_multi_news_summarization",
    "task618_amazonreview_summary_text_generation",
	"task668_extreme_abstract_summarization",

    "task629_dbpedia_14_classification",
    "task195_sentiment140_classification",
]

# Canonical repo id is hyphenated; underscore form is fallback.
dataset_candidates = [
    "Muennighoff/natural-instructions",
    "Muennighoff/natural_instructions",
]

last_error = None
full_dataset = None
for dataset_id in dataset_candidates:
    try:
        # Load default config and bypass strict split verification.
        full_dataset = load_dataset(
            dataset_id,
            split="train",
            verification_mode="no_checks",
        )
        break
    except (DatasetNotFoundError, ValueError) as err:
        last_error = err

if full_dataset is None:
    raise RuntimeError(
        f"Could not load any known dataset id: {dataset_candidates}. Last error: {last_error}"
    )

# Task ids are stored as data fields, not builder configs.
task_key_candidates = ["task_name", "task", "Task", "id", "task_id"]
task_key = next((k for k in task_key_candidates if k in full_dataset.column_names), None)

if task_key is None:
    raise RuntimeError(
        f"Could not find a task column. Available columns: {full_dataset.column_names}"
    )

raw_datasets = {}
for task in task_names:
    task_subset = full_dataset.filter(lambda ex: ex[task_key] == task)
    if len(task_subset) == 0 and task_key == "id":
        task_subset = full_dataset.filter(
            lambda ex: isinstance(ex["id"], str) and ex["id"].startswith(task)
        )
    raw_datasets[task] = task_subset

loaded_non_empty = sum(len(ds) > 0 for ds in raw_datasets.values())
print(f"Loaded {loaded_non_empty}/{len(task_names)} tasks with at least one example.")

Filter: 100%|██████████| 3082094/3082094 [00:27<00:00, 113297.85 examples/s]


Loaded 7/7 tasks with at least one example.


In [2]:
from datasets import load_dataset

probe = load_dataset(
    "Muennighoff/natural-instructions",
    split="train",
    verification_mode="no_checks",
)
print(probe)
print(probe.column_names)
print(probe[0])

Dataset({
    features: ['task_name', 'id', 'definition', 'inputs', 'targets'],
    num_rows: 3082094
})
['task_name', 'id', 'definition', 'inputs', 'targets']
{'task_name': 'task001_quoref_question_generation', 'id': 'task001-f44801d948324957abe71877d837d070', 'definition': "In this task, you're given passages that contain mentions of names of people, places, or things. Some of these mentions refer to the same person, place, or thing. Your job is to write questions that evaluate one's understanding of such references. Good questions are expected to link pronouns (she, her, him, his, their, etc.) or other mentions to people, places, or things to which they may refer. Do not ask questions that can be answered correctly without understanding the paragraph or having multiple answers. Avoid questions that do not link phrases referring to the same entity. For each of your questions, the answer should be one or more phrases in the paragraph, and it should be unambiguous.", 'inputs': 'Passage

In [11]:
from collections import Counter

# Show which requested tasks are missing and suggest close matches.
counts = Counter(full_dataset[task_key])

print(f"task key in use: {task_key}")
print("\nRequested task counts:")
for t in task_names:
    print(f"- {t}: {counts.get(t, 0)}")

missing = [t for t in task_names if counts.get(t, 0) == 0]
print(f"\nMissing tasks ({len(missing)}): {missing}")

if missing:
    all_tasks = list(counts.keys())
    print("\nClosest available task ids:")
    import difflib

    for t in missing:
        close = difflib.get_close_matches(t, all_tasks, n=5, cutoff=0.45)
        print(f"\n{t}")
        for c in close:
            print(f"  -> {c} (count={counts[c]})")

task key in use: task_name

Requested task counts:
- task075_squad1.1_answer_generation: 6983
- task568_circa_question_generation: 6500
- task1291_multi_news_summarization: 6527
- task618_amazonreview_summary_text_generation: 6500
- task668_extreme_abstract_summarization: 5398
- task629_dbpedia_14_classification: 3000
- task195_sentiment140_classification: 6500

Missing tasks (0): []
